In [0]:
fetch_date = dbutils.widgets.get('fetch_date')
charges_history_table = dbutils.widgets.get('charges_history_table')
target_table = dbutils.widgets.get('target_table')

In [0]:
display(
spark.sql(f"""       
-- CTE 1: Deduplicate and clean service code descriptions
WITH tmp10_dedup AS (
    SELECT 
        ClaimNumber,
        REGEXP_REPLACE(REGEXP_REPLACE(ServiceCodeDescription, '\\r', ''), '\\n', '') AS ServiceCodeDescription,
        ROW_NUMBER() OVER (
            PARTITION BY 
                ClaimNumber,
                REGEXP_REPLACE(REGEXP_REPLACE(ServiceCodeDescription, '\\r', ''), '\\n', '')
            ORDER BY Loaddate DESC
        ) AS rnb
    FROM {charges_history_table}
),

tmp10 AS (
    SELECT * FROM tmp10_dedup WHERE rnb = 1
),

-- CTE 2: Concatenate service code descriptions per claim
service_codes_aggregated AS (
    SELECT 
        ClaimNumber,
        CONCAT_WS(';', COLLECT_LIST(ServiceCodeDescription)) AS ServiceCodeDescription
    FROM tmp10
    GROUP BY ClaimNumber
)
-- Step 1: Update Program Name from aggregated service codes for CubHub records
MERGE INTO {target_table} AS target
USING (
    SELECT 
        ma.Invoice_Number,
        scd.ServiceCodeDescription AS Program_Name
    FROM {target_table} ma
    LEFT JOIN service_codes_aggregated scd
        ON ma.Invoice_Number = scd.ClaimNumber
    WHERE ma.reporting_date = CAST('{fetch_date}' AS DATE)
    AND ma.Source_System = 'CubHub'
) AS source
ON target.Invoice_Number = source.Invoice_Number
   AND target.reporting_date = CAST('{fetch_date}' AS DATE)

WHEN MATCHED THEN
    UPDATE SET target.Program_Name = source.Program_Name;
""")
)

In [0]:
display(
spark.sql(f"""  
-- Step 2: Update Active Workload and PCN
UPDATE {target_table}
SET 
    Active_Workload = CASE 
        WHEN Age_From_Bill_Date > 45 THEN 'Yes'
        WHEN Claim_Status = 'Denial' THEN 'Yes'
        ELSE 'No'
    END,
    PCN = CASE 
        WHEN Source_System = 'Bears' 
        THEN CONCAT(
            REGEXP_REPLACE(Client_Number, '-', ''),
            REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(Invoice_Number, '-', ''), ' ', ''), 'ADV', '')
        )
        ELSE Invoice_Number
    END
WHERE reporting_date = CAST('{fetch_date}' AS DATE);
""")
)

In [0]:
display(
spark.sql(f"""  
-- Step 2: Update Active Workload and PCN
UPDATE {target_table}
SET 
    Active_Workload = CASE 
        WHEN Age_From_Bill_Date > 45 THEN 'Yes'
        WHEN Claim_Status = 'Denial' THEN 'Yes'
        ELSE 'No'
    END,
    PCN = CASE 
        WHEN Source_System = 'Bears' 
        THEN CONCAT(
            REGEXP_REPLACE(Client_Number, '-', ''),
            REGEXP_REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(Invoice_Number, '-', ''), ' ', ''), 'ADV', '')
        )
        ELSE Invoice_Number
    END
WHERE reporting_date = CAST('{fetch_date}' AS DATE);
""")
)

In [0]:
display(
spark.sql(f""" 
-- Step 3: Update credit compliance and balance flags
UPDATE {target_table}
SET 
    day_150_credit_compliance_warning = CASE 
        WHEN Account_Balance < 0 
        AND Age_of_Credit_Balance >= 150
        AND Days_Untouched >= Age_of_Credit_Balance
        THEN 'Y'
        ELSE 'N'
    END,
    
    Small_Balance_Credit = CASE 
        -- Team 289 logic
        WHEN Reimbursement_Team = '289'
        AND Age_from_Bill_Date > 90
        AND Payer_Type NOT IN ('ATTORNEY', 'PRIV', 'LTC', 'TRUST', 'SELF PAY - HOME HEALTH')
        AND Account_Status <> 'Escheatment Identified'
        AND (
            -- HOSPICE - MEDICARE: -10 to 0
            (Payer_Type = 'HOSPICE - MEDICARE' 
             AND CAST(Credit_Reporting_Balance AS DOUBLE) >= -10 
             AND CAST(Credit_Reporting_Balance AS DOUBLE) < 0)
            -- MEDICARE: -10 to 0
            OR (Payer_Type = 'MEDICARE' 
                AND CAST(Credit_Reporting_Balance AS DOUBLE) >= -10 
                AND CAST(Credit_Reporting_Balance AS DOUBLE) < 0)
            -- MEDICARE - PART B: -10 to 0
            OR (Payer_Type = 'MEDICARE - PART B' 
                AND CAST(Credit_Reporting_Balance AS DOUBLE) >= -10 
                AND CAST(Credit_Reporting_Balance AS DOUBLE) < 0)
            -- PPS - NON MEDICARE: -50 to 0
            OR (Payer_Type = 'PPS - NON MEDICARE' 
                AND CAST(Credit_Reporting_Balance AS DOUBLE) >= -50 
                AND CAST(Credit_Reporting_Balance AS DOUBLE) < 0)
            -- Other payer types: -25 to 0
            OR (Payer_Type NOT IN ('PPS - NON MEDICARE', 'MEDICARE', 'HOSPICE - MEDICARE') 
                AND CAST(Credit_Reporting_Balance AS DOUBLE) >= -25 
                AND CAST(Credit_Reporting_Balance AS DOUBLE) < 0)
        )
        THEN 'Y'
        -- Non-289 teams logic
        WHEN Reimbursement_Team <> '289'
        AND Age_from_Bill_Date > 90
        AND CAST(Credit_Reporting_Balance AS DOUBLE) >= -25
        AND CAST(Credit_Reporting_Balance AS DOUBLE) < 0
        AND Payer_Type NOT IN ('ATTORNEY', 'PRIV', 'LTC', 'TRUST', 'SELF PAY - HOME HEALTH')
        AND Account_Status <> 'Escheatment Identified'
        THEN 'Y'
        ELSE 'N'
    END,
    
    six_year_lookback = CASE 
        WHEN Age_of_Credit_Balance > 2190
        AND Payer_Type NOT IN ('ATTORNEY', 'PRIV', 'LTC', 'TRUST', 'SELF PAY - HOME HEALTH')
        AND Account_Status <> 'Escheatment Identified'
        AND Days_Untouched > 180
        THEN 'Y'
        ELSE 'N'
    END
WHERE reporting_date = CAST('{fetch_date}' AS DATE);
""")
)